In [3]:
import pandas as pd
import numpy as np
import sklearn.linear_model as skl

class Logistic_Regression():
    def __init__(self):
        self.parameters = np.array([])
        self.bias = 0
        self.bias_vector = np.array([])
        self.learning_rate = 0.001
        self.beta_1 = 0.9
        self.beta_2 = 0.999
        self.max_iterations = 2500

    def fit(self, X, y, alpha):
        y = y.reshape(-1, 1)
        self.parameters = np.zeros(X.shape[1]).reshape(-1, 1)
        self.bias_vector =  np.full(shape=y.shape[0], fill_value=self.bias).reshape(-1, 1)
        first_moment_r = np.zeros(X.shape[1]).reshape(-1, 1)
        second_moment_r = np.zeros(X.shape[1]).reshape(-1, 1)
        first_moment_ur = 0
        second_moment_ur = 0
        for i in range(0, self.max_iterations):
            previous_parameters = self.parameters.copy()
            p = 1/(1+np.exp(-(X @ self.parameters + self.bias_vector)))
            grad_w = -(np.transpose(X) @ (y - p) -2 * X.shape[0] * alpha * self.parameters) * (1/X.shape[0])
            grad_b = -(np.mean(y - p))
            first_moment_r = self.beta_1 * first_moment_r + (1-self.beta_1) * grad_w 
            second_moment_r = self.beta_2 * second_moment_r + (1-self.beta_2) * np.square(grad_w)
            first_moment_ur = self.beta_1 * first_moment_ur + (1-self.beta_1) * grad_b
            second_moment_ur = self.beta_2 * second_moment_ur + (1-self.beta_2) * np.square(grad_b)
            m_hat_w = first_moment_r / (1 - self.beta_1 ** (i+1))
            v_hat_w = second_moment_r / (1 - self.beta_2 ** (i+1))
            m_hat_b = first_moment_ur / (1 - self.beta_1 ** (i+1))
            v_hat_b = second_moment_ur / (1 - self.beta_2 ** (i+1))
            self.parameters = self.parameters - self.learning_rate * (m_hat_w / (np.sqrt(v_hat_w) + 1e-8))
            self.bias = self.bias - self.learning_rate * (m_hat_b / (np.sqrt(v_hat_b) + 1e-8))
            self.bias_vector = np.full(shape=y.shape[0], fill_value=self.bias).reshape(-1, 1)
            if np.max(np.abs(self.parameters - previous_parameters)) < 1e-12:
                break
    def predict(self, X):
        bias_vector_prediction = np.full(shape=X.shape[0], fill_value=self.bias).reshape(-1, 1)
        y_hat = 1/(1+np.exp(-(X @ self.parameters + bias_vector_prediction)))
        return y_hat

def train_test_split(number_of_elements, testing_size):
    training_index = [i for i in range(0, number_of_elements)]
    testing_index = []
    for i in range(1, testing_size + 1):
        choice = np.random.choice(training_index)
        training_index.remove(choice)
        testing_index.append(choice)
    return training_index, testing_index

def kfold_cross_validation(number_of_folds, data_size):
    lists = []
    for i in range(0, number_of_folds):
        lists.append([])
    for j in range(0, data_size):
        list_index = j % number_of_folds
        lists[list_index].append(j)
    return lists

df = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\job_postings.csv')
X = df.drop(columns=['salary_year_avg'])
y = np.where(df['salary_year_avg'] > df['salary_year_avg'].median(), 1, 0)
numerical_columns = ['job_posted_month_sine', 'job_posted_year']
numerical_columns = [X.columns.get_loc(x) for x in numerical_columns]
X = X.to_numpy()

training_index, testing_index = train_test_split(len(y), 5000)
X_tuning = X[testing_index]
y_tuning = y[testing_index]
X = X[training_index]
y = y[training_index]

accuracies = {}
for alpha in np.logspace(-5, 2, 100):
    accuracies_one_fold = []
    for item in kfold_cross_validation(5, y_tuning.shape[0]):
        for column in numerical_columns:
            X_tuning[:, column] = (X_tuning[:, column] - np.delete(X_tuning, item, axis=0)[:, column].mean()) / np.delete(X_tuning, item, axis=0)[:, column].std()
        logistic_regression = Logistic_Regression()
        logistic_regression.fit(np.delete(X_tuning, item, axis=0), np.delete(y_tuning, item), alpha=alpha)
        predictions = logistic_regression.predict(X_tuning[item])
        predictions = np.array([1 if x > 0.5 else 0 for x in predictions])
        correct = 0
        total = y_tuning[item].shape[0]
        for predicted, true in zip(predictions, y_tuning[item]):
            if predicted == true:
                correct += 1
        accuracy = correct / total
        accuracies_one_fold.append(accuracy)
    accuracies.update({alpha: np.mean(accuracies_one_fold)})
best_alpha = max(accuracies, key=accuracies.get)
print(f'The Best Alpha Value Is {best_alpha:.4f}.')

accuracy_values = []
accuracy_values_skl = []
for item in kfold_cross_validation(5, y.shape[0]):
    for column in numerical_columns:
        X[:, column] = (X[:, column] - np.delete(X, item, axis=0)[column].mean()) / np.delete(X, item, axis=0)[column].std()
    total = y[item].shape[0]
    logistic_regression = Logistic_Regression()
    logistic_regression.fit(np.delete(X, item, axis=0), np.delete(y, item), alpha=best_alpha)
    predictions = logistic_regression.predict(X[item])
    predictions = [1 if x > 0.5 else 0 for x in predictions]
    correct = 0
    for predicted, true in zip(predictions, y[item]):
        if predicted == true:
            correct += 1
    accuracy = correct / total
    accuracy_values.append(accuracy)
    skl_logistic_regression = skl.LogisticRegressionCV(l1_ratios=([0]), scoring='accuracy', use_legacy_attributes=True, max_iter=10000)
    model = skl_logistic_regression.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions_skl = model.predict(X[item])
    correct_skl = 0
    for predicted, true in zip(predictions_skl, y[item]):
        if predicted == true:
            correct_skl += 1
    accuracy_skl = correct_skl / total
    accuracy_values_skl.append(accuracy_skl)
print(f'The Average Accuracy For Logistic Regression Is {np.mean(accuracy_values):.4%}.')
print(f'The Average Accuracy For Scikit-Learn\'s Logistic Regression Is {np.mean(accuracy_values_skl):.4%}.')

The Best Alpha Value Is 0.0003.
The Average Accuracy For Logistic Regression Is 73.7435%.
The Average Accuracy For Scikit-Learn's Logistic Regression Is 73.7735%.
